# The Embedding AtlasWhat shape is this corpus, as a distribution in 1024 dimensions?The chunks are embedded with `voyage-3.5` and searched by four retrieval strategies, butnobody has looked at the space itself. This notebook asks four questions:1. **Is the space healthy?** Or is everything crammed into a narrow cone where every   similarity score means less than it appears to?2. **How much of the Canon is repetition?** The Pali texts repeat stock formulas   (*pericopes*) verbatim by design. How much of the corpus is near-duplicate mass?3. **Does it cluster?** And if so, into what?4. **Can we trust the map?** MN 10 and DN 22 are near-identical texts. If the embeddings   are sound, they must land on top of each other.Nothing here changes retrieval. Findings become *hypotheses* for the eval-gated ladder.

In [ ]:
%load_ext autoreload%autoreload 2from dotenv import load_dotenvload_dotenv()import numpy as npimport pandas as pdimport plotly.express as pximport plotly.graph_objects as gofrom plotly.subplots import make_subplotsfrom atlas.loader import load, check_driftfrom atlas import geometry, structure, topics, pericopesvectors, df = load()check_drift()print(f"{len(df):,} chunks | {df['sutta_uid'].nunique()} suttas | {vectors.shape[1]} dimensions")print(df["nikaya"].value_counts().to_dict())print("all unit norm:", np.allclose(np.linalg.norm(vectors, axis=1), 1.0))

## 1. Is the space healthy?A baseline first, because raw numbers are meaningless without one. If 1024-dimensionalunit vectors were spread *isotropically* over the sphere, the mean cosine between tworandom chunks would be about **0**, and the mean vector would have length near **0**.Real embedding models are never isotropic. The question is how far from it this one is —because a narrow cone compresses the range every similarity score can occupy.

In [ ]:
aniso = geometry.anisotropy(vectors)print(f"mean pairwise cosine : {aniso['mean_pairwise_cosine']:.4f}   (isotropic would be ~0)")print(f"mean vector norm     : {aniso['mean_vector_norm']:.4f}   (isotropic would be ~0)")print()print(f"So the 'average chunk' direction accounts for "      f"{aniso['mean_vector_norm']**2:.1%} of a typical chunk vector.")

### Where does the signal actually live?Anisotropy alone does not tell us whether similarity is *useful*. That depends on the**gap** between related and unrelated pairs. If chunks from the same sutta score nohigher than chunks from different collections, the space carries no usable signal.

In [ ]:
dist = geometry.cosine_distributions(vectors, df, sample=300_000)fig = px.histogram(    dist, x="cosine", color="group", nbins=90, opacity=0.65,    barmode="overlay", histnorm="probability density",    title="Cosine similarity by relationship",    labels={"cosine": "cosine similarity", "group": ""},)fig.add_vline(x=aniso["mean_pairwise_cosine"], line_dash="dash", line_color="grey",              annotation_text="corpus mean")fig.update_layout(width=880, height=420)fig.show()summary = dist.groupby("group")["cosine"].agg(["mean", "std", "count"]).round(3)display(summary)print("\nsignal gap (within_sutta - cross_nikaya):",      round(summary.loc["within_sutta", "mean"] - summary.loc["cross_nikaya", "mean"], 3))

In [ ]:
curve = geometry.pca_curve(vectors)fig = go.Figure(go.Scatter(y=curve["cumulative"], mode="lines", name="cumulative variance"))for frac, dims in [(0.50, curve["dims_50"]), (0.90, curve["dims_90"]), (0.95, curve["dims_95"])]:    fig.add_hline(y=frac, line_dash="dot", line_color="lightgrey")    fig.add_annotation(x=dims, y=frac, text=f"{dims} dims", showarrow=True, arrowhead=1)fig.update_layout(title=f"Intrinsic dimensionality (of {vectors.shape[1]} nominal)",                  xaxis_title="principal components", yaxis_title="cumulative variance",                  width=880, height=420)fig.show()print(f"50% of variance in {curve['dims_50']} dims | "      f"90% in {curve['dims_90']} | 95% in {curve['dims_95']}")

### Hubs: chunks that get retrieved for everythingIn high dimensions a few points drift toward the centre of the cloud and end up inalmost every neighbourhood. They are retrieval parasites: they surface for queries theyhave nothing to do with. A healthy space has a flat k-occurrence distribution.

In [ ]:
counts = geometry.hubness(vectors, k=10)skew = geometry.hub_skew(counts)fig = px.histogram(x=counts, nbins=60,                   title=f"k-occurrence at k=10 (skew = {skew:.2f}; 0 is hub-free)",                   labels={"x": "times a chunk appears in another chunk's top-10"})fig.add_vline(x=10, line_dash="dash", line_color="grey", annotation_text="expected (k)")fig.update_layout(width=880, height=380, showlegend=False)fig.show()top_hubs = np.argsort(-counts)[:5]for rank, row in enumerate(top_hubs, 1):    r = df.iloc[row]    print(f"{rank}. [{counts[row]:>4} appearances] {r.sutta_uid} #{r.chunk_index} — {r.doc_title}")    print(f"   {r.chunk_text[:180].strip()}...\n")

In [ ]:
corr = geometry.length_vs_centrality(vectors, df)fig = px.scatter(    x=df["word_count"], y=vectors @ vectors.mean(axis=0),    opacity=0.5, trendline="ols", trendline_color_override="crimson",    title=f"Do long chunks drift to the centre?  (r = {corr:.3f})",    labels={"x": "words in chunk", "y": "similarity to corpus mean"},)fig.update_layout(width=880, height=420)fig.show()

## 2. How much of the Canon is repetition?The Pali Canon is formulaic on purpose. Stock passages — the jhāna formula, thesense-bases, "thus have I heard" — recur verbatim across hundreds of discourses, anartefact of centuries of oral transmission.This is not a data-quality problem to be cleaned. It is a structural property of thecorpus, and it has direct consequences for retrieval: if a query matches a stock formula,it matches it in *every* sutta that contains it.Chunks that are merely *adjacent* within one sutta are excluded — that is the chunkersplitting continuous prose, not genuine repetition.

In [ ]:
rows = []for threshold in (0.85, 0.90, 0.95):    _, stats = pericopes.families(vectors, df, threshold=threshold)    rows.append({"threshold": threshold, **stats})mass = pd.DataFrame(rows)mass["duplicate_mass"] = (mass["duplicate_mass"] * 100).round(1).astype(str) + "%"display(mass)

In [ ]:
labels, stats = pericopes.families(vectors, df, threshold=0.90)sizes = pd.Series(labels).value_counts()biggest = sizes[sizes > 1].head(3)for family_id, size in biggest.items():    members = df.iloc[np.flatnonzero(labels == family_id)]    suttas = ", ".join(sorted(members["sutta_uid"].unique())[:12])    print(f"=== family of {size} chunks across: {suttas} ===")    print(members.iloc[0]["chunk_text"][:300].strip(), "...\n")

### The correctness check: MN 10 against DN 22**MN 10** (*Satipaṭṭhāna Sutta*) and **DN 22** (*Mahāsatipaṭṭhāna Sutta*) are the samediscourse — DN 22 is MN 10 with the Four Noble Truths section expanded.This gives the map a free ground truth. If the embeddings are sound, each MN 10 chunkmust find its DN 22 twin at high cosine, and the alignment should be broadly monotonic.If it comes out scattered, nothing else in this notebook can be trusted.

In [ ]:
matches = pericopes.align(vectors, df, "mn10", "dn22")fig = px.scatter(    matches, x="mn10_index", y="dn22_index", color="cosine",    color_continuous_scale="Viridis", range_color=[0.5, 1.0],    title="MN 10 chunks aligned to their best match in DN 22",    labels={"mn10_index": "MN 10 chunk", "dn22_index": "best-matching DN 22 chunk"},)fig.update_traces(marker_size=11)fig.update_layout(width=760, height=520)fig.show()print(f"median cosine: {matches['cosine'].median():.3f} | "      f"min: {matches['cosine'].min():.3f} | "      f"above 0.8: {(matches['cosine'] > 0.8).mean():.0%}")print("rank correlation of the alignment:",      round(matches['mn10_index'].corr(matches['dn22_index'], method='spearman'), 3))

## 3. Does it cluster?Two rules govern what follows.**UMAP is for looking, not for deciding.** It distorts density by construction, soapparent gaps in the 2D picture can be pure artefact. Clustering therefore runs on thefull 1024 dimensions.**The projection is sensitive at this corpus size.** With fewer than ~2000 points,`n_neighbors` changes the picture substantially — so here are three, side by side. Anystructure that survives all three is probably real.

In [ ]:
layouts = {n: structure.project(vectors, n_neighbors=n) for n in (5, 15, 30)}fig = make_subplots(rows=1, cols=3, subplot_titles=[f"n_neighbors = {n}" for n in layouts])for col, (n, xy) in enumerate(layouts.items(), start=1):    for nikaya, colour in (("mn", "#4C78A8"), ("dn", "#F58518")):        m = (df["nikaya"] == nikaya).to_numpy()        fig.add_trace(            go.Scattergl(x=xy[m, 0], y=xy[m, 1], mode="markers", name=nikaya.upper(),                         marker=dict(size=4, color=colour, opacity=0.7),                         showlegend=(col == 1)),            row=1, col=col,        )fig.update_layout(title="The same corpus under three projection settings",                  width=1100, height=400)fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)fig.show()

In [ ]:
sweep = []for size in (10, 15, 25):    lab = structure.cluster(vectors, min_cluster_size=size)    sweep.append({        "min_cluster_size": size,        "clusters": len(set(lab.tolist()) - {-1}),        "noise": f"{(lab == -1).mean():.0%}",    })display(pd.DataFrame(sweep))

In [ ]:
MIN_CLUSTER_SIZE = 15          # chosen from the sweep above — see the prose cell belowcluster_labels = structure.cluster(vectors, min_cluster_size=MIN_CLUSTER_SIZE)ids, centres = structure.centroids(vectors, cluster_labels)exemplar_rows = structure.exemplars(vectors, cluster_labels)xy = layouts[15]import textwraphover = [    f"<b>{r.sutta_uid} #{r.chunk_index}</b> — {r.doc_title}<br>"    + "<br>".join(textwrap.wrap(r.chunk_text[:400].strip(), 60))    for r in df.itertuples()]plot = pd.DataFrame({    "x": xy[:, 0], "y": xy[:, 1],    "cluster": [f"{c}" if c != -1 else "noise" for c in cluster_labels],    "hover": hover,})fig = px.scatter(plot, x="x", y="y", color="cluster", hover_name="hover",                 title=f"The map — {len(ids)} clusters, coloured by cluster",                 category_orders={"cluster": [str(i) for i in ids] + ["noise"]})fig.update_traces(marker=dict(size=5, opacity=0.75), hovertemplate="%{hovertext}<extra></extra>")fig.add_trace(go.Scatter(    x=xy[list(exemplar_rows.values()), 0], y=xy[list(exemplar_rows.values()), 1],    mode="markers+text", marker=dict(size=13, color="black", symbol="x"),    text=[str(c) for c in exemplar_rows], textposition="top center",    name="centroid exemplar",))fig.update_layout(width=1000, height=650)fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)fig.show()

## 4. What are the clusters about?First **without** a domain stoplist. This pass is not a mistake — the formulaicscaffolding dominating every cluster is precisely the finding from §2, shown from adifferent angle.

In [ ]:
raw_terms = topics.ctfidf_terms(df["chunk_text"].tolist(), cluster_labels, top_n=8)for cluster_id, words in raw_terms.items():    print(f"cluster {cluster_id:>2}: {', '.join(words)}")

In [ ]:
terms = topics.ctfidf_terms(    df["chunk_text"].tolist(), cluster_labels, top_n=15, stopwords=topics.CANON_STOPWORDS)passages = {c: df.iloc[row]["chunk_text"] for c, row in exemplar_rows.items()}members = {    c: df.iloc[np.flatnonzero(cluster_labels == c)]["uuid"].tolist() for c in terms}named = topics.label_clusters(terms, passages, members)for cluster_id, label in named.items():    size = int((cluster_labels == cluster_id).sum())    print(f"cluster {cluster_id:>2} ({size:>3} chunks) — {label['name']}")    print(f"    {label['gloss']}")    print(f"    terms: {', '.join(terms[cluster_id][:8])}\n")

In [ ]:
plot["label"] = [    named[c]["name"] if c != -1 else "noise" for c in cluster_labels]fig = px.scatter(plot, x="x", y="y", color="label", hover_name="hover",                 title="The map, with topic labels")fig.update_traces(marker=dict(size=5, opacity=0.75), hovertemplate="%{hovertext}<extra></extra>")fig.update_layout(width=1050, height=680)fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)fig.show()

## What this says*Filled in after the run — including anything that came out negative or boring, whichstill counts as a result.*### Findings### Hypotheses for the eval gateMarked as hypotheses on purpose. Per the design, nothing here changes a retrievaldefault until it beats the incumbent on Recall@5 and MRR in `evals/`.